# MLP PyTorch: Random Forest e Rede Neural

Etapa 2: Modelagem com Redes Neurais — comparação da MLP (PyTorch) contra baselines lineares (Etapa 1) e um modelo ensemble (Random Forest).

| Modelo | Tipo | Seção |
|---|---|---|
| `RandomForestClassifier` | Ensemble (árvores) | 3 |
| `ChurnMLP` (PyTorch) | Rede neural | 4 |

**Métricas:** AUC-ROC, PR-AUC, Recall (churn), F1 (churn), Precisão (churn)  
**Validação:** Random Forest → StratifiedKFold (5 folds); MLP → split 70/15/15 com early stopping  
**Tracking:** MLflow com backend SQLite local (mesmo experimento dos baselines)

## 1. Setup

In [ ]:
import logging
import pathlib
import random
import subprocess
import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

SEED = 42
DATA_PATH = "../data/raw/telco-churn.csv"
MLFLOW_URI = "sqlite:///../mlruns.db"
EXPERIMENT_NAME = "churn-baselines"
N_FOLDS = 5
THRESHOLD = 0.4

# Seeds fixados para reprodutibilidade total
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

plt.rcParams["figure.figsize"] = (10, 4)
sns.set_theme(style="whitegrid")

log.info("Setup concluído. SEED=%d, FOLDS=%d, THRESHOLD=%.1f", SEED, N_FOLDS, THRESHOLD)

## 2. Dados

Mesmo carregamento e pré-processamento do notebook 02-baselines. Após o pré-processamento, realiza split estratificado 70/15/15 para o treinamento da MLP com early stopping.

**Por que split fixo para a MLP?** A MLP (PyTorch) precisa de um conjunto de validação separado para monitorar overfitting e ativar o early stopping. O Random Forest continua usando validação cruzada estratificada (mesma estratégia do notebook 02).

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

# Correções identificadas na EDA
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)

# Remover ID: não é feature preditiva
df = df.drop(columns=["customerID"])

log.info("Dataset carregado: %s", df.shape)
log.info("Churn rate: %.1f%%", df["Churn"].mean() * 100)

print(f"Shape: {df.shape}")
print(f"Churn rate: {df['Churn'].mean():.3f} ({df['Churn'].mean()*100:.1f}%)")

In [ ]:
TARGET = "Churn"
X = df.drop(columns=[TARGET])
y = df[TARGET]

# TotalCharges removida: colinearidade com tenure (r=+0.83) sem sinal independente
# gender e PhoneService removidas: Cramér's V ≈ 0 com o target
LOW_SIGNAL = ["gender", "PhoneService", "TotalCharges"]
X = X.drop(columns=LOW_SIGNAL)

# Interaction terms: Contract × tenure para cada tipo de contrato
X["monthly_x_tenure"]  = (X["Contract"] == "Month-to-month").astype(int) * X["tenure"]
X["one_year_x_tenure"] = (X["Contract"] == "One year").astype(int)       * X["tenure"]
X["two_year_x_tenure"] = (X["Contract"] == "Two year").astype(int)       * X["tenure"]

# tenure removido: completamente codificado nas três interações acima
X = X.drop(columns=["tenure"])

num_features = ["MonthlyCharges", "monthly_x_tenure", "one_year_x_tenure", "two_year_x_tenure"]
cat_features = [c for c in X.columns if c not in num_features]

print(f"Features numéricas ({len(num_features)}): {num_features}")
print(f"Features categóricas ({len(cat_features)}): {cat_features}")
print(f"Total de features: {X.shape[1]}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            num_features,
        ),
        (
            "cat",
            OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False),
            cat_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

log.info("Preprocessor definido: %d features numéricas, %d categóricas", len(num_features), len(cat_features))

In [ ]:
# Split 70/15/15 estratificado — usado exclusivamente para a MLP
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15 / 0.85, stratify=y_temp, random_state=SEED
)

log.info(
    "Splits: train=%d (%.1f%% churn), val=%d (%.1f%% churn), test=%d (%.1f%% churn)",
    len(X_train), y_train.mean() * 100,
    len(X_val),   y_val.mean()   * 100,
    len(X_test),  y_test.mean()  * 100,
)

print(f"Train: {len(X_train)} amostras ({y_train.mean()*100:.1f}% churn)")
print(f"Val:   {len(X_val)} amostras ({y_val.mean()*100:.1f}% churn)")
print(f"Test:  {len(X_test)} amostras ({y_test.mean()*100:.1f}% churn)")

In [ ]:
# Preprocessor: fit apenas no treino, transform no val e test
preprocessor.fit(X_train)
X_train_t = preprocessor.transform(X_train)
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)

log.info("Shapes pós-transformação: train=%s, val=%s, test=%s",
         X_train_t.shape, X_val_t.shape, X_test_t.shape)
print(f"Número de features após OHE: {X_train_t.shape[1]}")

In [ ]:
def to_tensor(X_arr, y_series):
    """Converte array numpy e Series pandas para tensores PyTorch."""
    X_t = torch.tensor(X_arr, dtype=torch.float32)
    y_t = torch.tensor(y_series.values, dtype=torch.float32).unsqueeze(1)
    return X_t, y_t


X_train_pt, y_train_pt = to_tensor(X_train_t, y_train)
X_val_pt,   y_val_pt   = to_tensor(X_val_t,   y_val)
X_test_pt,  y_test_pt  = to_tensor(X_test_t,  y_test)

log.info(
    "Tensores criados: X_train=%s, X_val=%s, X_test=%s",
    tuple(X_train_pt.shape), tuple(X_val_pt.shape), tuple(X_test_pt.shape),
)
print(f"dtype X: {X_train_pt.dtype} | dtype y: {y_train_pt.dtype}")

## 3. Random Forest

`RandomForestClassifier` com `class_weight='balanced'` para compensar o desbalanceamento de classes.  
Avaliado com `StratifiedKFold` (5 folds) sobre o **dataset completo**, exatamente como os baselines do notebook 02.

O split 70/15/15 é reservado exclusivamente para a MLP — o Random Forest não precisa de val set, pois o CV já estima o desempenho fora da amostra.

In [ ]:
CV = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def _make_threshold_scorer(metric_fn):
    """Retorna scorer com assinatura (estimator, X, y) usando threshold customizado."""
    def scorer(estimator, X, y):
        proba = estimator.predict_proba(X)[:, 1]
        y_pred = (proba >= THRESHOLD).astype(int)
        return metric_fn(y, y_pred, zero_division=0)
    return scorer


recall_t04    = _make_threshold_scorer(recall_score)
f1_t04        = _make_threshold_scorer(f1_score)
precision_t04 = _make_threshold_scorer(precision_score)


def evaluate_cv(pipeline, X, y, model_name):
    """Avalia pipeline com cross-validation estratificado. Retorna (métricas_resumo, cv_results_brutos)."""
    log.info("Avaliando '%s' com %d folds (threshold=%.1f)...", model_name, N_FOLDS, THRESHOLD)

    scoring = {
        "roc_auc": "roc_auc",
        "average_precision": "average_precision",
        "recall": recall_t04,
        "f1": f1_t04,
        "precision": precision_t04,
    }

    cv_results = cross_validate(
        pipeline,
        X,
        y,
        cv=CV,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1,
    )

    metrics = {
        "roc_auc_mean": float(cv_results["test_roc_auc"].mean()),
        "roc_auc_std": float(cv_results["test_roc_auc"].std()),
        "pr_auc_mean": float(cv_results["test_average_precision"].mean()),
        "pr_auc_std": float(cv_results["test_average_precision"].std()),
        "recall_mean": float(np.nan_to_num(cv_results["test_recall"].mean())),
        "recall_std": float(np.nan_to_num(cv_results["test_recall"].std())),
        "f1_mean": float(np.nan_to_num(cv_results["test_f1"].mean())),
        "f1_std": float(np.nan_to_num(cv_results["test_f1"].std())),
        "precision_mean": float(np.nan_to_num(cv_results["test_precision"].mean())),
        "precision_std": float(np.nan_to_num(cv_results["test_precision"].std())),
    }

    log.info(
        "%s → AUC-ROC=%.3f | PR-AUC=%.3f | Recall=%.3f | F1=%.3f | Precision=%.3f",
        model_name,
        metrics["roc_auc_mean"],
        metrics["pr_auc_mean"],
        metrics["recall_mean"],
        metrics["f1_mean"],
        metrics["precision_mean"],
    )
    return metrics, cv_results

In [ ]:
# Random Forest usa preprocessor novo (sem fit prévio) — o CV cuida do fit em cada fold
rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            num_features,
        ),
        (
            "cat",
            OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False),
            cat_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

rf_pipe = Pipeline([
    ("preprocessor", rf_preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=SEED,
        ),
    ),
])

rf_metrics, rf_cv = evaluate_cv(rf_pipe, X, y, "Random Forest")

In [ ]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.sklearn.autolog(disable=True)

mlflow_dataset = mlflow.data.from_pandas(
    df,
    source=DATA_PATH,
    name="telco-churn",
    targets="Churn",
)

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
    ).decode().strip()
except Exception:
    GIT_COMMIT = "unknown"

log.info("MLflow tracking URI: %s", MLFLOW_URI)
log.info("Experimento: %s", EXPERIMENT_NAME)
log.info("Git commit: %s", GIT_COMMIT)
print(f"MLflow UI: mlflow ui --backend-store-uri {MLFLOW_URI}")

In [ ]:
params_rf = {
    "model": "RandomForestClassifier",
    "n_estimators": 100,
    "max_depth": None,
    "class_weight": "balanced",
    "threshold": THRESHOLD,
    "n_folds": N_FOLDS,
    "seed": SEED,
}

# Treina no dataset completo para salvar o artefato e inferir a assinatura
rf_pipe.fit(X, y)
signature_rf = mlflow.models.infer_signature(X, rf_pipe.predict_proba(X)[:, 1])

with mlflow.start_run(run_name="random-forest-balanced"):
    mlflow.set_tags({"stage": "candidate", "model_type": "ensemble", "git_commit": GIT_COMMIT})
    mlflow.log_input(mlflow_dataset, context="training")
    mlflow.log_params(params_rf)
    mlflow.log_metrics(rf_metrics)
    for fold_idx, val in enumerate(rf_cv["test_roc_auc"]):
        mlflow.log_metric("roc_auc_fold", val, step=fold_idx)
    for fold_idx, val in enumerate(rf_cv["test_average_precision"]):
        mlflow.log_metric("pr_auc_fold", val, step=fold_idx)
    mlflow.sklearn.log_model(
        rf_pipe,
        name="model",
        input_example=X.head(3),
        signature=signature_rf,
    )
    run_id_rf = mlflow.active_run().info.run_id

log.info("Run registrado: %s", run_id_rf)

## 4. MLP (PyTorch)

Rede neural `ChurnMLP` com duas camadas ocultas (64 → 32 → 1), ReLU e Dropout.

**Decisões de design:**
- `BCEWithLogitsLoss` com `pos_weight` para compensar desbalanceamento de classes (mesmo princípio do `class_weight='balanced'` do sklearn)
- Early stopping com paciência de 10 épocas para evitar overfitting
- Adam com weight decay (L2 regularization) para generalização
- Threshold 0.4: mesmo threshold dos baselines, priorizando recall

**Avaliação:** conjunto de teste fixo (15% do dataset), separado antes do treinamento.

In [ ]:
class ChurnMLP(nn.Module):
    """MLP para classificação binária de churn.

    Arquitetura: Linear(n_features, 64) → ReLU → Dropout
                 → Linear(64, 32) → ReLU → Dropout
                 → Linear(32, 1)  (logit)
    """

    def __init__(self, n_features: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32),         nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


n_features = X_train_t.shape[1]
model = ChurnMLP(n_features=n_features, dropout=0.3)

log.info("Arquitetura MLP: %d features → 64 → 32 → 1", n_features)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal de parâmetros: {total_params:,}")

In [ ]:
# pos_weight: compensa o desbalanceamento de classes na loss
n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

log.info(
    "Classe positiva (churn): %d amostras | Negativa: %d | pos_weight=%.2f",
    n_pos, n_neg, pos_weight.item(),
)
print(f"pos_weight: {pos_weight.item():.3f} (n_neg/n_pos = {n_neg}/{n_pos})")

In [ ]:
BATCH_SIZE = 64
MAX_EPOCHS = 100
PATIENCE   = 10

train_ds = TensorDataset(X_train_pt, y_train_pt)
_g = torch.Generator()
_g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=_g)

log.info(
    "DataLoader: %d amostras, batch_size=%d → %d batches/época",
    len(train_ds), BATCH_SIZE, len(train_loader),
)

In [ ]:
params_mlp = {
    "model": "MLP",
    "hidden_dims": "64-32",
    "dropout_rate": 0.3,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "pos_weight": float(pos_weight.item()),
    "threshold": THRESHOLD,
    "train_size": len(X_train),
    "val_size": len(X_val),
    "test_size": len(X_test),
    "seed": SEED,
}

best_val_loss     = float("inf")
best_epoch_idx    = 0
epochs_no_improve = 0
best_weights      = None
train_losses      = []
val_losses        = []

with mlflow.start_run(run_name="mlp-pytorch") as run:
    mlflow.set_tags({"stage": "candidate", "model_type": "neural_network", "git_commit": GIT_COMMIT})
    mlflow.log_input(mlflow_dataset, context="training")
    mlflow.log_params(params_mlp)

    # --- Loop de treinamento com early stopping ---
    for epoch in range(MAX_EPOCHS):
        # Treino
        model.train()
        batch_losses = []
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        train_loss = float(np.mean(batch_losses))

        # Validação
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_pt)
            val_loss   = criterion(val_logits, y_val_pt).item()
            val_proba  = torch.sigmoid(val_logits).numpy().ravel()
            val_auc    = roc_auc_score(y_val, val_proba)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss",   val_loss,   step=epoch)
        mlflow.log_metric("val_roc_auc", val_auc,   step=epoch)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss     = val_loss
            best_epoch_idx    = epoch
            epochs_no_improve = 0
            best_weights      = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                log.info("Early stopping na época %d (paciência=%d)", epoch + 1, PATIENCE)
                break

    if best_weights is not None:
        model.load_state_dict(best_weights)
    log.info("Treinamento concluído. Melhor época: %d | Melhor val_loss: %.4f", best_epoch_idx, best_val_loss)

    # --- Avaliação no conjunto de teste ---
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_pt)
        test_proba  = torch.sigmoid(test_logits).numpy().ravel()
        test_pred   = (test_proba >= THRESHOLD).astype(int)

    mlp_metrics = {
        "roc_auc":  roc_auc_score(y_test, test_proba),
        "pr_auc":   average_precision_score(y_test, test_proba),
        "recall":   recall_score(y_test, test_pred),
        "f1":       f1_score(y_test, test_pred),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "best_epoch": float(best_epoch_idx),
        "best_val_loss": float(best_val_loss),
    }
    mlflow.log_metrics(mlp_metrics)

    log.info(
        "MLP (teste) → AUC-ROC=%.3f | PR-AUC=%.3f | Recall=%.3f | F1=%.3f | Precision=%.3f",
        mlp_metrics["roc_auc"], mlp_metrics["pr_auc"],
        mlp_metrics["recall"],  mlp_metrics["f1"], mlp_metrics["precision"],
    )

    # --- Artefato 1: curva de treinamento ---
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_losses, label="train_loss", color="steelblue")
    ax.plot(val_losses,   label="val_loss",   color="tomato")
    ax.axvline(
        best_epoch_idx, color="green", linestyle="--", alpha=0.7,
        label=f"best epoch ({best_epoch_idx})",
    )
    ax.set_xlabel("Época")
    ax.set_ylabel("Loss")
    ax.set_title("Curva de Treinamento da MLP (BCEWithLogitsLoss)")
    ax.legend()
    plt.tight_layout()
    mlflow.log_figure(fig, "mlp_training_curve.png")
    plt.show()

    # --- Artefato 2: pesos do melhor checkpoint ---
    models_dir = pathlib.Path("../models")
    models_dir.mkdir(exist_ok=True)
    checkpoint_path = models_dir / "mlp_best.pt"
    torch.save(model.state_dict(), checkpoint_path)
    mlflow.log_artifact(str(checkpoint_path), artifact_path="checkpoints")
    log.info("Checkpoint salvo em %s", checkpoint_path)

    # --- Artefato 3: modelo PyTorch com assinatura ---
    signature = infer_signature(X_test_t[:3], test_proba[:3])
    mlflow.pytorch.log_model(
        model,
        name="model",
        input_example=X_test_t[:3],
        signature=signature,
    )

    # --- Artefato 4: notebook como artefato ---
    notebook_path = pathlib.Path("03-mlp.ipynb")
    if notebook_path.exists():
        mlflow.log_artifact(str(notebook_path))
        log.info("Notebook logado como artefato no MLflow")

    run_id_mlp = run.info.run_id

log.info("Run MLP registrado: %s", run_id_mlp)

## 5. Tabela Comparativa de Resultados

Comparação entre todos os modelos treinados até aqui. Baselines da Etapa 1 (DummyClassifier, Logistic Regression) são incluídos para referência.

> **Nota metodológica:** Random Forest e baselines usam CV (5 folds, mean ± std). A MLP usa avaliação em conjunto de teste fixo (single point estimate), portanto as comparações diretas devem considerar essa diferença na estimativa de variância.

In [ ]:
# Importar métricas dos baselines (Etapa 1) do MLflow
client = mlflow.MlflowClient()

_exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
_runs = client.search_runs(
    experiment_ids=[_exp.experiment_id],
    filter_string="",
    order_by=["start_time ASC"],
)

# Mapeia run_name → métricas
runs_by_name = {}
for _run in _runs:
    _name = _run.data.tags.get("mlflow.runName", _run.info.run_id)
    runs_by_name[_name] = _run.data.metrics

log.info("Runs encontrados: %s", list(runs_by_name.keys()))

# Nomes dos runs de baseline conforme registrados no notebook 02
BASELINE_RUNS = {
    "Dummy (most_frequent)": "dummy-most-frequent",
    "Dummy (stratified)":    "dummy-stratified",
    "Logistic Regression":   "logistic-regression",
}

TARGETS = {
    "AUC-ROC":  0.80,
    "PR-AUC":   0.60,
    "Recall":   0.70,
    "F1":       0.60,
    "Precisão": 0.55,
}

rows = []

# Baselines da Etapa 1 (métricas do MLflow)
for display_name, run_name in BASELINE_RUNS.items():
    m = runs_by_name.get(run_name, {})
    rows.append({
        "Modelo":   display_name,
        "AUC-ROC":  f"{m.get('roc_auc_mean', float('nan')):.3f}",
        "PR-AUC":   f"{m.get('pr_auc_mean',  float('nan')):.3f}",
        "Recall":   f"{m.get('recall_mean',  float('nan')):.3f}",
        "F1":       f"{m.get('f1_mean',       float('nan')):.3f}",
        "Precisão": f"{m.get('precision_mean', float('nan')):.3f}",
    })

# Random Forest — métricas do CV (calculadas acima neste notebook)
rows.append({
    "Modelo":   "Random Forest (CV)",
    "AUC-ROC":  f"{rf_metrics['roc_auc_mean']:.3f} ± {rf_metrics['roc_auc_std']:.3f}",
    "PR-AUC":   f"{rf_metrics['pr_auc_mean']:.3f} ± {rf_metrics['pr_auc_std']:.3f}",
    "Recall":   f"{rf_metrics['recall_mean']:.3f} ± {rf_metrics['recall_std']:.3f}",
    "F1":       f"{rf_metrics['f1_mean']:.3f} ± {rf_metrics['f1_std']:.3f}",
    "Precisão": f"{rf_metrics['precision_mean']:.3f} ± {rf_metrics['precision_std']:.3f}",
})

# MLP — avaliação pontual no test set
rows.append({
    "Modelo":   "MLP PyTorch (test)",
    "AUC-ROC":  f"{mlp_metrics['roc_auc']:.3f}",
    "PR-AUC":   f"{mlp_metrics['pr_auc']:.3f}",
    "Recall":   f"{mlp_metrics['recall']:.3f}",
    "F1":       f"{mlp_metrics['f1']:.3f}",
    "Precisão": f"{mlp_metrics['precision']:.3f}",
})

comparison_df = pd.DataFrame(rows).set_index("Modelo")

print(f"{'='*72}")
print(f"TABELA COMPARATIVA: TODOS OS MODELOS — Etapas 1 e 2 (threshold={THRESHOLD})")
print(f"{'='*72}")
display(comparison_df)

print("\nMetas mínimas (ML Canvas):")
for label, target in TARGETS.items():
    print(f"  {label:<10}: >= {target:.2f}")


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

# Curvas ROC e PR: Random Forest vs MLP no mesmo conjunto de teste
# RF: predict_proba no X_test (preprocessamento embutido no pipeline)
rf_test_proba = rf_pipe.predict_proba(X_test)[:, 1]
churn_rate = float(y_test.mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Curva ROC ---
ax_roc = axes[0]
RocCurveDisplay.from_predictions(
    y_test, rf_test_proba,
    name=f"Random Forest (AUC={rf_metrics['roc_auc_mean']:.3f} CV)",
    ax=ax_roc, color="steelblue",
)
RocCurveDisplay.from_predictions(
    y_test, test_proba,
    name=f"MLP PyTorch (AUC={mlp_metrics['roc_auc']:.3f} test)",
    ax=ax_roc, color="tomato",
)
ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Aleatório")
ax_roc.set_title(f"Curva ROC — RF vs MLP (threshold={THRESHOLD})")
ax_roc.legend(loc="lower right", fontsize=9)

# --- Curva PR ---
ax_pr = axes[1]
PrecisionRecallDisplay.from_predictions(
    y_test, rf_test_proba,
    name=f"Random Forest (PR-AUC={rf_metrics['pr_auc_mean']:.3f} CV)",
    ax=ax_pr, color="steelblue",
)
PrecisionRecallDisplay.from_predictions(
    y_test, test_proba,
    name=f"MLP PyTorch (PR-AUC={mlp_metrics['pr_auc']:.3f} test)",
    ax=ax_pr, color="tomato",
)
ax_pr.axhline(churn_rate, color="k", linestyle="--", alpha=0.4,
              label=f"Baseline aleatório (churn rate={churn_rate:.2f})")
ax_pr.set_title(f"Curva Precision-Recall — RF vs MLP (threshold={THRESHOLD})")
ax_pr.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

log.info(
    "Curvas ROC/PR geradas: RF AUC=%.3f (CV), MLP AUC=%.3f (test)",
    rf_metrics['roc_auc_mean'], mlp_metrics['roc_auc'],
)


## 6. Análise de Custo: Falsos Positivos vs Falsos Negativos

**Contexto de negócio:**

| Erro | Situação | Consequência | Custo estimado |
|------|----------|--------------|----------------|
| Falso Negativo (FN) | Modelo não detecta churn real | Cliente cancela sem intervenção | Alto — receita perdida |
| Falso Positivo (FP) | Modelo aciona retenção desnecessariamente | Campanha para quem não cancelaria | Baixo — custo de campanha |

Como FN >> FP em impacto financeiro, o modelo deve **priorizar recall** (minimizar FN).  
O threshold=0.4 (em vez do padrão 0.5) foi escolhido exatamente por esse motivo: aceita mais FP para capturar mais churn real.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred,
    display_labels=["Não churn", "Churn"],
    colorbar=False,
    cmap="Blues",
    ax=ax,
)
ax.set_title(f"Matriz de Confusão — MLP PyTorch (threshold={THRESHOLD})")
plt.tight_layout()
plt.show()

_tn = int(((test_pred == 0) & (y_test == 0)).sum())
_fp = int(((test_pred == 1) & (y_test == 0)).sum())
_fn = int(((test_pred == 0) & (y_test == 1)).sum())
_tp = int(((test_pred == 1) & (y_test == 1)).sum())
log.info("Confusão MLP: TN=%d, FP=%d, FN=%d, TP=%d", _tn, _fp, _fn, _tp)
print(f"TN={_tn}  FP={_fp}  FN={_fn}  TP={_tp}")


In [ ]:
# Análise de sensibilidade ao threshold
thresholds = [0.3, 0.4, 0.5]
thresh_rows = []
for t in thresholds:
    pred_t = (test_proba >= t).astype(int)
    thresh_rows.append({
        "Threshold": t,
        "Precisão":  precision_score(y_test, pred_t, zero_division=0),
        "Recall":    recall_score(y_test, pred_t),
        "F1":        f1_score(y_test, pred_t),
        "FP": int(((pred_t == 1) & (y_test == 0)).sum()),
        "FN": int(((pred_t == 0) & (y_test == 1)).sum()),
    })

threshold_df = pd.DataFrame(thresh_rows).set_index("Threshold")
display(threshold_df.round(3))

log.info("Análise de threshold concluída para %s", thresholds)


### Interpretação de Negócio

- **Threshold 0.3** — recall máximo, mas com mais FPs: mais campanhas desnecessárias.
- **Threshold 0.4** *(escolhido)* — equilíbrio: recall elevado com precisão aceitável. Justificado pelo custo assimétrico: perder um cliente tem impacto maior do que o custo de uma campanha de retenção desnecessária.
- **Threshold 0.5** — threshold padrão: favorece precisão, mas perde mais clientes reais (mais FNs).

**Decisão:** manter threshold=0.4 para o modelo em produção. A revisão do threshold deve ser feita trimestralmente conforme o custo real de campanha vs. LTV médio do cliente.
